In [0]:

from pyspark.sql.functions import (
    col, current_timestamp, lit, row_number, monotonically_increasing_id,
    when, datediff, year, quarter, month, dayofmonth, dayofweek, weekofyear,
    date_format, concat, to_date, expr, sum as _sum, avg, min as _min, max as _max,
    countDistinct, round as spark_round, coalesce
)
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, StringType, DateType, BooleanType, DoubleType, TimestampType
from datetime import datetime, timedelta

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS 03_gold.final_report")

In [0]:
start_date = datetime(2022, 1, 1)
end_date = datetime(2026, 12, 31)
date_list = [start_date + timedelta(days=x) for x in range((end_date - start_date).days + 1)]
from pyspark.sql import Row
date_rows = [Row(date=d) for d in date_list]
df_dates = spark.createDataFrame(date_rows)
df_dim_date = df_dates.select(
    date_format(col("date"), "yyyyMMdd").cast(IntegerType()).alias("date_key"),
    col("date").alias("date"),
    year(col("date")).alias("year"),
    quarter(col("date")).alias("quarter"),
    concat(lit("Q"), quarter(col("date")), lit(" "), year(col("date"))).alias("quarter_name"),
    month(col("date")).alias("month"),
    date_format(col("date"), "MMMM").alias("month_name"),
    date_format(col("date"), "MMM yyyy").alias("month_year"),
    weekofyear(col("date")).alias("week_of_year"),
    dayofmonth(col("date")).alias("day_of_month"),
    dayofweek(col("date")).alias("day_of_week"),
    date_format(col("date"), "EEEE").alias("day_name"),
    when(dayofweek(col("date")).isin(1, 7), True).otherwise(False).alias("is_weekend"),
    when(month(col("date")) >= 4, year(col("date"))).otherwise(year(col("date")) - 1).alias("fiscal_year"),
    when(month(col("date")).between(4, 6), 1)
        .when(month(col("date")).between(7, 9), 2)
        .when(month(col("date")).between(10, 12), 3)
        .otherwise(4).alias("fiscal_quarter"),
    current_timestamp().alias("processed_timestamp")
)
df_dim_date.write.format("delta").mode("overwrite").saveAsTable("03_gold.final_report.dim_date")


In [0]:
df_customer_silver = spark.table("02_silver.clean.customer")
window_spec = Window.orderBy("customer_id")
df_dim_customer = df_customer_silver.withColumn(
    "customer_key",
    row_number().over(window_spec))
df_dim_customer = df_dim_customer.withColumn(
    "is_active",
    when(col("is_active").isin("Y", "Yes", "1", "yes", "y", "true", "True", "TRUE", True), True)
    .when(col("is_active").isin("N", "No", "0", "no", "n", "false", "False", "FALSE", False), False)
    .otherwise(None).cast(BooleanType())
)
df_dim_customer = df_dim_customer.select(
    col("customer_key").cast(IntegerType()),  # Surrogate key
    col("customer_id").cast(IntegerType()),    # Business key
    col("customer_name"),
    col("country"),
    col("industry_type"),
    col("account_created_date"),
    col("is_active"),
    col("account_created_date").alias("effective_start_date"),
    to_date(lit("9999-12-31")).alias("effective_end_date"),
    lit(True).alias("is_current"),
    
    current_timestamp().alias("processed_timestamp")
)
df_dim_customer.write.format("delta").mode("overwrite").saveAsTable("03_gold.final_report.dim_customer")


In [0]:
df_employee_silver = spark.table("02_silver.clean.employee")
window_spec = Window.orderBy("employee_id")
df_dim_employee = df_employee_silver.withColumn(
    "employee_key",
    row_number().over(window_spec)
)
df_dim_employee = df_dim_employee.select(
    col("employee_key").cast(IntegerType()),  # Surrogate key
    col("employee_id").cast(IntegerType()),    # Business key
    col("employee_name"),
    col("role"),
    col("region"),
    col("hire_date"),
    col("is_active_flag"),
    col("last_update"),
    current_timestamp().alias("processed_timestamp")
)
df_dim_employee.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("03_gold.final_report.dim_employee")

In [0]:
df_product_silver = spark.table("02_silver.clean.product")
window_spec = Window.orderBy("product_id")
df_dim_product = df_product_silver.withColumn(
    "product_key",
    row_number().over(window_spec)
)
df_dim_product = df_dim_product.select(
    col("product_key").cast(IntegerType()),  # Surrogate key
    col("product_id").cast(IntegerType()),    # Business key
    col("product_name"),
    col("plan_name"),
    col("billing_cycle"),
    col("list_price"),
    col("is_active"),
    col("created_date"),
    current_timestamp().alias("processed_timestamp")
)
df_dim_product.write.format("delta").mode("overwrite").saveAsTable("03_gold.final_report.dim_product")

In [0]:
df_fx_silver = spark.table("02_silver.clean.fx_rate")
window_spec = Window.orderBy("currency_code", "effective_date")
df_dim_currency = df_fx_silver.withColumn(
    "currency_key",
    row_number().over(window_spec)
)
df_dim_currency = df_dim_currency.select(
    col("currency_key").cast(IntegerType()),  # Surrogate key
    col("currency_code"),
    col("fx_rate_to_gbp"),
    col("effective_date"),
    current_timestamp().alias("processed_timestamp")
)
df_dim_currency.write.format("delta").mode("overwrite").saveAsTable("03_gold.final_report.dim_currency")

In [0]:

df_fact = spark.table("02_silver.clean.opportunity")
df_dim_customer = spark.table("03_gold.final_report.dim_customer").select("customer_key", "customer_id")
df_dim_employee = spark.table("03_gold.final_report.dim_employee").select("employee_key", "employee_id")
df_dim_product = spark.table("03_gold.final_report.dim_product").select("product_key", "product_id")
df_fact = df_fact \
    .join(df_dim_customer, df_fact.customer_id == df_dim_customer.customer_id, "left") \
    .join(df_dim_employee, df_fact.employee_id == df_dim_employee.employee_id, "left") \
    .join(df_dim_product, df_fact.product_id == df_dim_product.product_id, "left")
df_fact = df_fact.withColumn(
    "start_date_key",
    date_format(col("start_date"), "yyyyMMdd").cast(IntegerType())
).withColumn(
    "end_date_key",
    date_format(col("end_date"), "yyyyMMdd").cast(IntegerType())
).withColumn(
    "created_date_key",
    date_format(col("created_timestamp"), "yyyyMMdd").cast(IntegerType())
)

df_fact = df_fact.withColumn(
    "contract_duration_days",
    datediff(col("end_date"), col("start_date"))
)
df_fact = df_fact.withColumn(
    "monthly_recurring_revenue",
    when(col("contract_term") == "Annual", col("revenue_amount") / 12.0)
    .otherwise(col("revenue_amount"))
)
df_fact = df_fact.withColumn(
    "annual_recurring_revenue",
    col("monthly_recurring_revenue") * 12.0
)

window_spec = Window.orderBy("opportunity_id")
df_fact = df_fact.withColumn(
    "opportunity_key",
    row_number().over(window_spec)
)
df_fact_opportunity = df_fact.select(
    col("opportunity_key").cast(IntegerType()),
    col("opportunity_id").cast(IntegerType()),
    col("customer_key"),
    col("product_key"),
    col("employee_key"),
    col("start_date_key"),
    col("end_date_key"),
    col("created_date_key"),
    col("revenue_amount"),
    col("monthly_recurring_revenue"),
    col("annual_recurring_revenue"),
    col("contract_duration_days"),
    col("contract_term"),
    col("close_status"),
    col("start_date"),
    col("end_date"),
    col("created_timestamp"),
    current_timestamp().alias("processed_timestamp")
)
df_fact_opportunity.write.format("delta").mode("overwrite").saveAsTable("03_gold.final_report.fact_opportunity")

In [0]:
df_curated = spark.table("03_gold.final_report.fact_opportunity")
df_curated = df_curated.filter(col("opportunity_id").isNotNull())
df_customer_dim = spark.table("03_gold.final_report.dim_customer").filter(col("is_current") == True)
df_curated = df_curated.join(
    df_customer_dim.select(
        col("customer_key"),
        col("customer_id"),
        col("customer_name"),
        col("country"),
        col("industry_type"),
        col("is_active").alias("customer_is_active"),
        col("account_created_date")
    ),
    "customer_key",
    "left"
)
df_employee_dim = spark.table("03_gold.final_report.dim_employee")
df_curated = df_curated.join(
    df_employee_dim.select(
        col("employee_key"),
        col("employee_id"),
        col("employee_name"),
        col("role").alias("employee_role"),
        col("region").alias("employee_region"),
        col("is_active_flag").cast(BooleanType()).alias("employee_is_active") 
    ),
    "employee_key",
    "left"
) 
df_product_dim = spark.table("03_gold.final_report.dim_product")
df_curated = df_curated.join(
    df_product_dim.select(
        col("product_key"),
        col("product_id"),
        col("product_name"),
        col("plan_name"),
        col("billing_cycle"),
        col("list_price"),
        col("is_active").alias("product_is_active")
    ),
    "product_key",
    "left"
)
df_date_dim = spark.table("03_gold.final_report.dim_date")
df_curated = df_curated.join(
    df_date_dim.select(
        col("date_key").alias("start_date_key_join"),
        col("date").alias("start_date_full"),
        col("year").alias("start_year"),
        col("quarter").alias("start_quarter"),
        col("month").alias("start_month"),
        col("fiscal_year").alias("start_fiscal_year"),
        col("fiscal_quarter").alias("start_fiscal_quarter")
    ),
    df_curated.start_date_key == col("start_date_key_join"),
    "left"
).drop("start_date_key_join")
df_curated = df_curated.join(
    df_date_dim.select(
        col("date_key").alias("end_date_key_join"),
        col("date").alias("end_date_full"),
        col("year").alias("end_year"),
        col("quarter").alias("end_quarter"),
        col("month").alias("end_month")
    ),
    df_curated.end_date_key == col("end_date_key_join"),
    "left"
).drop("end_date_key_join")
df_curated = df_curated.join(
    df_date_dim.select(
        col("date_key").alias("created_date_key_join"),
        col("date").alias("created_date_full"),
        col("year").alias("created_year"),
        col("quarter").alias("created_quarter"),
        col("month").alias("created_month")
    ),
    df_curated.created_date_key == col("created_date_key_join"),
    "left"
).drop("created_date_key_join")
df_curated_final = df_curated.select(
    col("opportunity_key"),
    col("opportunity_id"),
    col("close_status"),
    col("contract_term"),
    col("customer_id"),
    col("customer_name"),
    col("country"),
    col("industry_type"),
    col("customer_is_active"),
    col("account_created_date"),
    col("employee_id"),
    col("employee_name"),
    col("employee_role"),
    col("employee_region"),
    col("employee_is_active"),
    col("product_id"),
    col("product_name"),
    col("plan_name"),
    col("billing_cycle"),
    col("list_price"),
    col("product_is_active"),
    col("start_date"),
    col("start_year"),
    col("start_quarter"),
    col("start_month"),
    col("start_fiscal_year"),
    col("start_fiscal_quarter"),
    col("end_date"),
    col("end_year"),
    col("end_quarter"),
    col("end_month"),
    col("created_timestamp"),
    col("created_year"),
    col("created_quarter"),
    col("created_month"),
    col("revenue_amount"),
    col("monthly_recurring_revenue"),
    col("annual_recurring_revenue"),
    col("contract_duration_days"),
    current_timestamp().alias("curated_timestamp")
)
df_curated_final.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("03_gold.final_report.curated_opportunity")